# Capstone Project 1: Heart Disease Risk — an End-to-End Classification Pipeline

This project ties together the **Statistical Foundations** module (hypothesis testing,
correlation, train/test discipline, bias-variance) and the **Machine Learning
Fundamentals** module (logistic regression, random forests, evaluation metrics) into
one worked example, on the UCI Heart Disease dataset already used in
`5. MLOps/2. End-to-End ML/heart_disease.ipynb`.

**The question:** given a patient's clinical measurements, predict whether they have
heart disease — and along the way, use hypothesis tests to check *which* measurements
actually differ between patients who do and don't have it, rather than just trusting a
model's feature importances blindly.

### Workflow

1. Load and explore the data
2. Statistical tests: which features differ significantly between the two groups?
3. Train/test split, with the discipline from Notebook 10 of the stats module
4. Baseline, logistic regression, and random forest models
5. Evaluation: confusion matrix, ROC-AUC, calibration
6. Feature importance, cross-checked against the hypothesis tests from step 2
7. A written recommendation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay,
                             brier_score_loss)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance

rng = np.random.default_rng(42)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
SKF = StratifiedKFold(5, shuffle=True, random_state=0)

## 1. Load and explore

In [ ]:
df = pd.read_csv("../5. MLOps/2. End-to-End ML/data/heart_disease_cleaned_2.csv", index_col=0)
print(df.shape)
df.head()

In [ ]:
print(df.dtypes)
print("\nmissing values:\n", df.isnull().sum())
print(f"\ntarget prevalence: {df['target'].mean():.3f}")

`target` is 1 for heart disease present, 0 for absent. The columns follow the standard
Cleveland heart-disease naming: `cp` (chest pain type), `trestbps` (resting blood
pressure), `chol` (cholesterol), `fbs` (fasting blood sugar > 120 mg/dl), `restecg`
(resting ECG results), `thalach` (max heart rate achieved), `exang` (exercise-induced
angina), `slope`, `ca` (number of major vessels coloured by fluoroscopy), `thal`
(thalassemia result).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
numeric_cols = ["age", "trestbps", "chol", "thalach"]
for ax, col in zip(axes.ravel(), numeric_cols):
    sns.histplot(data=df, x=col, hue="target", kde=True, ax=ax, palette="coolwarm",
                bins=25, alpha=0.6)
    ax.set_title(col)
sns.countplot(data=df, x="cp", hue="target", ax=axes[1, 2], palette="coolwarm")
axes[1, 2].set_title("chest pain type")
plt.tight_layout()
plt.show()

## 2. Which features actually differ, statistically?

For each numeric feature, run a two-sample t-test (Notebook 7 of the stats module)
comparing patients with and without heart disease. This is the discipline that
separates "the model liked this feature" from "this feature has a real, testable
association with the outcome" — and it is worth doing *before* modelling, since it also
surfaces data-quality problems (a feature with an implausible p-value is worth a second
look).

In [ ]:
numeric_features = ["age", "trestbps", "chol", "thalach"]
rows = []
for col in numeric_features:
    group0 = df.loc[df.target == 0, col]
    group1 = df.loc[df.target == 1, col]
    result = stats.ttest_ind(group1, group0, equal_var=False)   # Welch's t-test
    pooled_sd = np.sqrt(((len(group1)-1)*group1.var(ddof=1) + (len(group0)-1)*group0.var(ddof=1))
                        / (len(group1) + len(group0) - 2))
    cohens_d = (group1.mean() - group0.mean()) / pooled_sd
    rows.append({"feature": col, "mean_disease": group1.mean(), "mean_no_disease": group0.mean(),
                "t_stat": result.statistic, "p_value": result.pvalue, "cohens_d": cohens_d})

pd.DataFrame(rows).round(4)

In [ ]:
# Categorical features: chi-square test of independence (stats Notebook 8)
categorical_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
rows_c = []
for col in categorical_features:
    table = pd.crosstab(df[col], df["target"])
    chi2, p, dof, expected = stats.chi2_contingency(table)
    cramers_v = np.sqrt(chi2 / (len(df) * (min(table.shape) - 1)))
    rows_c.append({"feature": col, "chi2": chi2, "p_value": p, "cramers_v": cramers_v})

chi_results = pd.DataFrame(rows_c).sort_values("cramers_v", ascending=False)
chi_results.round(4)

`cp` (chest pain type), `thal`, `exang`, `ca` and `sex` show the strongest associations
with the target (largest Cramér's V, smallest p-values). We should expect these to also
come out as important in the trained models below — if they do not, that is worth
investigating rather than shrugging off.

## 3. Train/test split

Following the discipline from stats Notebook 10: split before touching the test set
again, stratify on the target since we care about a roughly-balanced classification
task, and never look at test performance until the final evaluation.

In [ ]:
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,
                                                     stratify=y)
print(f"train: {len(y_train)} rows, {y_train.mean():.3f} prevalence")
print(f"test : {len(y_test)} rows, {y_test.mean():.3f} prevalence")

## 4. Models: baseline, logistic regression, random forest

In [ ]:
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_train, y_train)
forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0).fit(X_train, y_train)

print(f"{'model':<20}{'CV ROC-AUC':>13}{'test ROC-AUC':>15}{'test accuracy':>16}")
for name, model in [("baseline", baseline), ("logistic regression", logit), ("random forest", forest)]:
    try:
        cv_auc = cross_val_score(model, X_train, y_train, cv=SKF, scoring="roc_auc").mean()
        test_proba = model.predict_proba(X_test)[:, 1]
        test_auc = roc_auc_score(y_test, test_proba)
    except Exception:
        cv_auc = test_auc = float("nan")
    test_acc = model.score(X_test, y_test)
    print(f"{name:<20}{cv_auc:>13.4f}{test_auc:>15.4f}{test_acc:>16.4f}")

## 5. Evaluation of the chosen model

In [ ]:
best_model = logit if roc_auc_score(y_test, logit.predict_proba(X_test)[:, 1]) >= \
                     roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1]) else forest
best_name = "logistic regression" if best_model is logit else "random forest"
print(f"Selected model: {best_name}")

pred = best_model.predict(X_test)
proba = best_model.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                       display_labels=["no disease", "disease"]).plot(ax=axes[0], cmap="Blues",
                                                                      colorbar=False)
axes[0].set_title("Confusion matrix")

fpr, tpr, _ = roc_curve(y_test, proba)
axes[1].plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC={roc_auc_score(y_test, proba):.3f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate")
axes[1].set_title("ROC curve"); axes[1].legend(fontsize=8)

frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=8, strategy="quantile")
axes[2].plot([0, 1], [0, 1], "k--", lw=1)
axes[2].plot(mean_pred, frac_pos, "o-", color="seagreen")
axes[2].set_title(f"Calibration (Brier {brier_score_loss(y_test, proba):.4f})")
axes[2].set_xlabel("predicted probability"); axes[2].set_ylabel("observed frequency")
plt.tight_layout()
plt.show()

print(classification_report(y_test, pred, target_names=["no disease", "disease"]))

## 6. Feature importance, cross-checked against the hypothesis tests

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, n_repeats=30, random_state=0,
                              scoring="roc_auc")
importance = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
importance.plot(kind="barh", ax=ax, color="steelblue")
ax.invert_yaxis()
ax.set_title("Permutation importance (drop in ROC-AUC)")
plt.tight_layout()
plt.show()

top_stat_features = set(chi_results.nsmallest(4, "p_value")["feature"]) | \
                    set(pd.DataFrame(rows).nsmallest(2, "p_value")["feature"])
top_model_features = set(importance.head(5).index)
print(f"Top features by hypothesis test : {sorted(top_stat_features)}")
print(f"Top features by model importance: {sorted(top_model_features)}")
print(f"Overlap: {sorted(top_stat_features & top_model_features)}")

## 7. Recommendation

Write a short recommendation here once you've run the cells above: which model you
would deploy, what its ROC-AUC and calibration look like, which features drive its
predictions, and — critically — what you would tell a clinician about the model's
limitations (this is a demonstration on cleaned, moderate-sized data; a real deployment
needs a much larger validation cohort, a fairness audit across demographic subgroups,
and sign-off from a clinical stakeholder before it informs any real decision).

In [ ]:
print("=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Deployed model      : {best_name}")
print(f"Test ROC-AUC         : {roc_auc_score(y_test, proba):.4f}")
print(f"Test accuracy        : {best_model.score(X_test, y_test):.4f}")
print(f"Baseline accuracy    : {baseline.score(X_test, y_test):.4f}")
print(f"Top 3 features       : {list(importance.head(3).index)}")
print(f"Statistically significant features (p<0.05, chi-square): "
      f"{chi_results[chi_results.p_value < 0.05]['feature'].tolist()}")
print("=" * 70)